In [1]:
# Google Colab / local setup
import os, sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CODE_ROOT = Path("/content/drive/MyDrive/msc project/code")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
        str(CODE_ROOT), 'catboost==1.2.10', 'rapidfuzz==3.14.3',
    ])
else:
    CODE_ROOT = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    )
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
CODE = CODE_ROOT

# 02e — All-model sentiment policy calibration

## Methodology overview

This notebook continues Notebook 02d and uses the same nine model families, four arms (**No sentiment**, **DeBERTa**, **LLM-matched** and **LLM-full**), DZ55/DZ65/DZ75, 180-day history, fixed 15-minute horizon and 5 bps per side. It does not repeat the 2024 blocking-fold or raw-sentiment experiment.

All three sentiment arms see the same sources; they differ only in the scorer and in how each weights what it reads. **DeBERTa against LLM-full** is the layer-versus-layer comparison, and **LLM-matched against LLM-full** isolates how much of any difference is the scalar rather than the structure.

For each January–June 2025 month, each Model/Arm/DZ is fitted only on the preceding 180 days and predicts that month. The six causal prediction blocks evaluate 33 pre-declared policies: 11 confidence thresholds crossed with TP/SL 150/75, 150/100 and 200/100 bps; hold is one M15 bar. Selection ranks economically adequate policies (at least 50 trades, 15 long, 15 short and four positive months) by Sortino, then Sharpe and net return.

The selected policy is frozen and applied to the exact Notebook 02d July-2025 fit and July-2025–March-2026 predictions. One-minute candles are used only to resolve TP/SL execution after the M15 signal. The forward span cannot change the policy, and Q2-2026 is evaluated separately in Notebook 07.

### 1. Load and validate artifacts
The checks require every Model/Arm root and one unique policy and forward row per Model/Arm/DZ.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)

CODE_ROOT = Path.cwd().resolve()
if CODE_ROOT.name == 'notebooks':
    CODE_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from experiments.all_model_sentiment_policy import ARMS, DEFAULT_ROOT, WIDTHS, validate_policy_model_artifacts
from experiments.all_model_sentiment_policy_scoreboard import build_scoreboards
from experiments.all_model_sentiment_scoreboard import build_scoreboards as build_raw_scoreboards
from experiments.raw_hold_control import MODEL_LABELS, MODEL_NAMES

ROOT = DEFAULT_ROOT
for arm in ARMS:
    for model_name in MODEL_NAMES:
        validate_policy_model_artifacts(ROOT / arm / model_name, model_name=model_name)

tables = build_scoreboards(ROOT)
policies = tables['policies']
economics = tables['economics']
raw_economics = build_raw_scoreboards()['economics']
keys = ['sentiment_arm', 'model_name', 'width_bps']
expected = len(ARMS) * 9 * len(WIDTHS)
assert len(policies) == expected and policies[keys].drop_duplicates().shape[0] == expected
assert len(economics) == expected and economics[keys].drop_duplicates().shape[0] == expected
assert len(raw_economics) == expected and raw_economics[keys].drop_duplicates().shape[0] == expected
assert set(policies['width_bps']) == set(WIDTHS)

### 2. H1 2025 policy selection
Each row is the policy selected from six causal monthly prediction blocks; these are calibration results, not forward results.

In [3]:
from IPython.display import Markdown
h1 = policies[[
    'model_name', 'Arm', 'width_bps', 'tau', 'tp_bps', 'sl_bps',
    'hold_minutes', 'trades', 'pooled_net', 'pooled_sortino',
    'pooled_sharpe', 'positive_segments',
]].copy()
h1['model_name'] = h1['model_name'].map(MODEL_LABELS)
h1.columns = [
    'Model', 'Sentiment', 'DZ', 'Threshold', 'TP', 'SL', 'Hold (min)',
    'Trades', 'Net return', 'Sortino', 'Sharpe', 'Positive months',
]
h1 = h1.sort_values(
    ['Sortino', 'Sharpe', 'Net return'], ascending=False, ignore_index=True
)
display(Markdown('Method: Each Model/Arm/DZ policy is selected on six causal H1 monthly prediction blocks.'))
display(h1.round(3))
display(Markdown('H1 results are calibration evidence, not an independent profitability test.'))

Method: Each Model/Arm/DZ policy is selected on six causal H1 monthly prediction blocks.

,Model,Sentiment,DZ,Threshold,TP,SL,Hold (min),Trades,Net return,Sortino,Sharpe,Positive months
0,MLP,DeBERTa,55,0.80,200,100,15,66,0.065,3.314,1.646,4
1,Logistic Regression,LLM-matched,65,0.80,150,75,15,52,0.046,2.436,1.367,4
2,MLP,LLM-matched,55,0.80,200,100,15,58,0.043,2.316,1.237,3
3,Linear SVM,LLM-matched,65,0.00,150,75,15,19,0.027,1.793,0.974,2
4,Logistic Regression,LLM-full,65,0.80,150,75,15,66,0.028,1.163,0.709,4
5,LSTM,No sentiment,55,0.75,200,100,15,79,0.026,1.150,0.659,4
6,Linear SVM,DeBERTa,75,0.00,150,100,15,12,0.008,0.470,0.297,2
7,LSTM,No sentiment,65,0.75,200,100,15,136,0.007,0.253,0.158,4
8,Linear SVM,No sentiment,75,0.00,200,100,15,17,0.004,0.233,0.142,2
9,Linear SVM,LLM-full,55,0.00,150,75,15,30,0.002,0.115,0.075,1


H1 results are calibration evidence, not an independent profitability test.

### 3. Frozen-forward economics
The H1-selected threshold and TP/SL are replayed unchanged on July 2025–March 2026; no row is removed because the final research decision is manual.

In [4]:
from IPython.display import Markdown
forward = economics[[
    'model_name', 'Arm', 'width_bps', 'tau', 'tp_bps', 'sl_bps',
    'max_hold', 'trades', 'net_return', 'sortino', 'sharpe', 'positive_months',
]].copy()
forward['model_name'] = forward['model_name'].map(MODEL_LABELS)
forward['max_hold'] = forward['max_hold'] * 15
forward.columns = [
    'Model', 'Sentiment', 'DZ', 'Threshold', 'TP', 'SL', 'Hold (min)',
    'Trades', 'Net return', 'Sortino', 'Sharpe', 'Positive months',
]
forward = forward.sort_values(['Sortino', 'Sharpe', 'Net return'], ascending=False).reset_index(drop=True)
display(Markdown('Method: H1-selected thresholds and barriers remain unchanged throughout the nine-month forward period.'))
display(forward.round(3))
display(Markdown('Policy calibration creates selective winners but does not make sentiment consistently beneficial.'))

Method: H1-selected thresholds and barriers remain unchanged throughout the nine-month forward period.

,Model,Sentiment,DZ,Threshold,TP,SL,Hold (min),Trades,Net return,Sortino,Sharpe,Positive months
0,Linear SVM,No sentiment,75,0.00,200,100,15,30,0.055,2.409,1.304,6
1,LSTM,No sentiment,55,0.75,200,100,15,54,0.036,1.568,0.859,5
2,Linear SVM,No sentiment,65,0.00,150,75,15,51,0.001,0.035,0.024,4
3,Linear SVM,LLM-full,55,0.00,150,75,15,58,-0.013,-0.412,-0.277,4
4,Linear SVM,LLM-full,75,0.00,200,100,15,19,-0.010,-0.422,-0.279,2
5,Linear SVM,LLM-full,65,0.00,150,75,15,30,-0.018,-0.752,-0.525,1
6,Linear SVM,LLM-matched,55,0.00,150,75,15,92,-0.040,-0.999,-0.699,3
7,XGBoost,No sentiment,65,0.80,150,75,15,131,-0.054,-1.368,-0.923,4
8,CatBoost,DeBERTa,65,0.70,200,100,15,38,-0.039,-1.400,-1.067,4
9,Linear SVM,No sentiment,55,0.00,150,100,15,80,-0.070,-1.650,-1.202,3


Policy calibration creates selective winners but does not make sentiment consistently beneficial.

### 4. Five uncalibrated model-family leaders
This is the policy-free Notebook 02d control: each family keeps its best Sentiment/DZ row by raw-forward Sortino, then Sharpe and net return.

In [5]:
from IPython.display import Markdown
raw_leaders = raw_economics.sort_values(
    ['sortino', 'sharpe', 'net_return'], ascending=False
).drop_duplicates('model_name').head(5).copy()
raw_leaders['model_name'] = raw_leaders['model_name'].map(MODEL_LABELS)
raw_leaders = raw_leaders[[
    'model_name', 'Arm', 'width_bps', 'hold_minutes', 'trades',
    'net_return', 'sortino', 'sharpe', 'positive_months',
]]
raw_leaders.columns = [
    'Model', 'Sentiment', 'DZ', 'Hold (min)', 'Trades', 'Net return',
    'Sortino', 'Sharpe', 'Positive months',
]
display(Markdown('Method: The five raw controls retain one best Arm/DZ row per family before policy calibration.'))
display(raw_leaders.round(3))
display(Markdown('These rows provide the policy-free reference for the calibrated comparison.'))

Method: The five raw controls retain one best Arm/DZ row per family before policy calibration.

,Model,Sentiment,DZ,Hold (min),Trades,Net return,Sortino,Sharpe,Positive months
103,Linear SVM,No sentiment,75,15,40,0.075,1.838,1.201,6
104,XGBoost,DeBERTa,75,15,585,-0.620,-5.964,-4.736,1
72,CatBoost,DeBERTa,75,15,273,-0.479,-6.091,-5.014,1
96,Random Forest,DeBERTa,75,15,2437,-2.724,-16.599,-13.210,0
87,Logistic Regression,No sentiment,75,15,4479,-4.473,-22.733,-18.388,0


These rows provide the policy-free reference for the calibrated comparison.

### 5. Five calibrated model-family leaders
Each family keeps its best Arm/DZ/policy row; the table then reports the top five families by forward Sortino, with Sharpe and net return as tie-breakers. Because a family shortlist can only show one row per model, the per-arm leaders are reported separately below — otherwise an arm whose best row shares a family with another arm's would be invisible.

In [6]:
from IPython.display import Markdown
leaders = economics.sort_values(
    ['sortino', 'sharpe', 'net_return'], ascending=False
).drop_duplicates('model_name').head(5).copy()
leaders['model_name'] = leaders['model_name'].map(MODEL_LABELS)
leaders['max_hold'] = leaders['max_hold'] * 15
leaders = leaders[[
    'model_name', 'Arm', 'width_bps', 'tau', 'tp_bps', 'sl_bps',
    'max_hold', 'trades', 'net_return', 'sortino', 'sharpe', 'positive_months',
]]
leaders.columns = [
    'Model', 'Sentiment', 'DZ', 'Threshold', 'TP', 'SL', 'Hold (min)',
    'Trades', 'Net return', 'Sortino', 'Sharpe', 'Positive months',
]
display(Markdown('Method: One calibrated row per family is ranked descriptively on forward Sortino, Sharpe and Net.'))
display(leaders.round(3))
display(Markdown('These leaders are development candidates, not Q2-selected winners.'))

per_arm = economics.sort_values(
    ['sortino', 'sharpe', 'net_return'], ascending=False
).drop_duplicates('sentiment_arm').copy()
per_arm['model_name'] = per_arm['model_name'].map(MODEL_LABELS)
per_arm['max_hold'] = per_arm['max_hold'] * 15
per_arm['gross_bps'] = per_arm['gross_return'] / per_arm['trades'].clip(lower=1) * 1e4
per_arm = per_arm[[
    'Arm', 'model_name', 'width_bps', 'tau', 'tp_bps', 'sl_bps', 'max_hold',
    'trades', 'net_return', 'sortino', 'sharpe', 'positive_months', 'gross_bps',
]]
per_arm.columns = [
    'Arm', 'Model', 'DZ', 'Threshold', 'TP', 'SL', 'Hold (min)', 'Trades',
    'Net return', 'Sortino', 'Sharpe', 'Positive months', 'Gross bps/trade',
]
display(Markdown('Method: Each sentiment arm retains its strongest calibrated row under the same ranking rule.'))
display(per_arm.round(3).reset_index(drop=True))
display(Markdown('Best-row results do not show consistent paired improvement across families.'))

paired = economics.pivot_table(
    index=['model_name', 'width_bps'], columns='sentiment_arm', values='net_return')
cells = len(paired)
for arm, label in (('classic', 'DeBERTa'), ('llm', 'LLM-matched'), ('llm_full', 'LLM-full')):
    if arm not in paired:
        continue
    delta = paired[arm] - paired['none']
if {'llm_full', 'classic'} <= set(paired.columns):
    delta = paired['llm_full'] - paired['classic']

raw_by_model = raw_economics.sort_values(
    ['sortino', 'sharpe', 'net_return'], ascending=False
).drop_duplicates('model_name').set_index('model_name')
calibrated_by_model = economics.sort_values(
    ['sortino', 'sharpe', 'net_return'], ascending=False
).drop_duplicates('model_name').set_index('model_name')
improvement = calibrated_by_model[['sortino', 'net_return']].join(
    raw_by_model[['sortino', 'net_return']],
    how='inner',
    lsuffix='_calibrated',
    rsuffix='_raw',
)
assert len(improvement) == 9
sortino_improvements = int(
    (improvement['sortino_calibrated'] > improvement['sortino_raw']).sum()
)
net_improvements = int(
    (improvement['net_return_calibrated'] > improvement['net_return_raw']).sum()
)

Method: One calibrated row per family is ranked descriptively on forward Sortino, Sharpe and Net.

,Model,Sentiment,DZ,Threshold,TP,SL,Hold (min),Trades,Net return,Sortino,Sharpe,Positive months
103,Linear SVM,No sentiment,75,0.00,200,100,15,30,0.055,2.409,1.304,6
19,LSTM,No sentiment,55,0.75,200,100,15,54,0.036,1.568,0.859,5
71,XGBoost,No sentiment,65,0.80,150,75,15,131,-0.054,-1.368,-0.923,4
36,CatBoost,DeBERTa,65,0.70,200,100,15,38,-0.039,-1.400,-1.067,4
47,GRU,No sentiment,65,0.75,150,75,15,96,-0.053,-1.924,-1.400,2


These leaders are development candidates, not Q2-selected winners.

Method: Each sentiment arm retains its strongest calibrated row under the same ranking rule.

,Arm,Model,DZ,Threshold,TP,SL,Hold (min),Trades,Net return,Sortino,Sharpe,Positive months,Gross bps/trade
0,No sentiment,Linear SVM,75,0.0,200,100,15,30,0.055,2.409,1.304,6,28.182
1,LLM-full,Linear SVM,55,0.0,150,75,15,58,-0.013,-0.412,-0.277,4,7.750
2,LLM-matched,Linear SVM,55,0.0,150,75,15,92,-0.040,-0.999,-0.699,3,5.673
3,DeBERTa,CatBoost,65,0.7,200,100,15,38,-0.039,-1.400,-1.067,4,-0.215


Best-row results do not show consistent paired improvement across families.

### 6. Handoff
The five rows above are forward-development candidates, not final lockbox winners. They may proceed to ensemble testing, while Q2-2026 is evaluated separately in Notebook 07 until the final specification is frozen.